In [1]:
# ============================================================================
# ЯЧЕЙКА 1: Импорт библиотек и настройка Spark сессии
# ============================================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import tempfile
import os

from hypex.dataset import Dataset, TargetRole, TreatmentRole
from hypex.dataset.backends import SparkDataset

# Создаем Spark сессию
spark = (SparkSession.builder 
    .appName("HypEx-Spark-Tests") 
    .master("local[*]") 
    .config("spark.driver.memory", "4g") 
    .config("spark.sql.shuffle.partitions", "4") 
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

26/02/24 19:34:45 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.240.116.115 instead (on interface en0)
26/02/24 19:34:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 19:34:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.8


In [2]:
import pyspark

In [3]:
# ============================================================================
# ЯЧЕЙКА 2: Вспомогательные функции и фикстуры для тестов
# ============================================================================
def create_test_df(spark_session, data=None, schema=None):
    """Создает тестовый DataFrame с простыми данными"""
    if data is None:
        data = [
            (1, "Alice", 25, 3.14, True),
            (2, "Bob", 30, 2.71, False),
            (3, "Charlie", 35, 1.41, True),
            (4, "Diana", 28, 9.81, False),
            (5, "Eve", 22, 0.57, True)
        ]
    if schema is None:
        schema = StructType([
            StructField("id", IntegerType(), True),
            StructField("name", StringType(), True),
            StructField("age", IntegerType(), True),
            StructField("score", DoubleType(), True),
            StructField("is_active", BooleanType(), True)
        ])
    return spark_session.createDataFrame(data, schema)

def assert_df_equal(df1, df2, ignore_order=False):
    """Проверяет равенство двух Spark DataFrame"""
    if ignore_order:
        # Сортируем по всем колонкам для сравнения
        cols = df1.columns
        df1_sorted = df1.orderBy(cols)
        df2_sorted = df2.orderBy(cols)
        assert df1_sorted.collect() == df2_sorted.collect()
    else:
        assert df1.collect() == df2.collect()

print("✓ Вспомогательные функции загружены")

✓ Вспомогательные функции загружены


In [4]:
# ============================================================================
# ЯЧЕЙКА 3: Тесты конструктора __init__
# ============================================================================
print("=== Тесты __init__ ===")

# Тест 3.1: Инициализация с Spark DataFrame
print("\n[TEST 3.1] Инициализация с Spark DataFrame")
test_df = create_test_df(spark)
ds = SparkDataset(data=test_df, session=spark)
assert ds.data is not None, "data не должен быть None"
assert ds.data.count() == 5, "Должно быть 5 строк"
assert len(ds.data.columns) == 5, "Должно быть 5 колонок"
print("✓ Passed")

# Тест 3.2: Инициализация с pandas DataFrame
print("\n[TEST 3.2] Инициализация с pandas DataFrame")
pdf = pd.DataFrame({
    "id": [1, 2, 3],
    "value": ["a", "b", "c"]
})
ds_pd = SparkDataset(data=pdf, session=spark)
assert ds_pd.data.count() == 3, "Должно быть 3 строки"
assert set(ds_pd.data.columns) == {"id", "value"}, "Колонки должны совпадать"
print("✓ Passed")

# Тест 3.3: Инициализация с dict
print("\n[TEST 3.3] Инициализация с dict")
dict_data = {"data": {"id": [1, 2], "name": ["X", "Y"]}}
ds_dict = SparkDataset(data=dict_data, session=spark)
assert ds_dict.data.count() == 2, "Должно быть 2 строки"
print("✓ Passed")

# Тест 3.4: Инициализация с dict с index
print("\n[TEST 3.4] Инициализация с dict с index")
dict_with_index = {
    "data": {"id": [1, 2], "name": ["X", "Y"]},
    "index": ["a", "b"]
}
ds_dict_idx = SparkDataset(data=dict_with_index, session=spark)
assert ds_dict_idx.data.count() == 2, "Должно быть 2 строки"
print("✓ Passed")

# Тест 3.5: Инициализация с None (пустой DataFrame)
print("\n[TEST 3.5] Инициализация с None")
ds_empty = SparkDataset(data=None, session=spark)
assert ds_empty.data is not None, "data не должен быть None"
assert ds_empty.data.count() == 0, "Должен быть пустой DataFrame"
print("✓ Passed")

# Тест 3.6: Ошибка при отсутствии session
print("\n[TEST 3.6] Ошибка при отсутствии session")
try:
    ds_no_session = SparkDataset(data=test_df, session=None)
    assert False, "Должна быть ошибка ValueError"
except ValueError as e:
    assert "Session not set" in str(e)
    print("✓ Passed: ValueError raised correctly")

# Тест 3.7: Ошибка при неверном типе session
print("\n[TEST 3.7] Ошибка при неверном типе session")
try:
    ds_bad_session = SparkDataset(data=test_df, session="not_a_session")
    assert False, "Должна быть ошибка TypeError"
except TypeError as e:
    assert "Session must be an instance of SparkSession" in str(e)
    print("✓ Passed: TypeError raised correctly")

print("\n✅ Все тесты __init__ пройдены!")

=== Тесты __init__ ===

[TEST 3.1] Инициализация с Spark DataFrame


✓ Passed

[TEST 3.2] Инициализация с pandas DataFrame
✓ Passed

[TEST 3.3] Инициализация с dict
✓ Passed

[TEST 3.4] Инициализация с dict с index
✓ Passed

[TEST 3.5] Инициализация с None
✓ Passed

[TEST 3.6] Ошибка при отсутствии session
✓ Passed: ValueError raised correctly

[TEST 3.7] Ошибка при неверном типе session
✓ Passed: TypeError raised correctly

✅ Все тесты __init__ пройдены!


In [5]:
# ============================================================================
# ЯЧЕЙКА 4: Тесты _read_file (статический метод)
# ============================================================================
print("=== Тесты _read_file ===")

# Создаем временную директорию для тестовых файлов
temp_dir = tempfile.mkdtemp()

# Тест 4.1: Чтение CSV файла
print("\n[TEST 4.1] Чтение CSV файла")
csv_path = Path(temp_dir) / "test.csv"
pd.DataFrame({"id": [1, 2], "name": ["A", "B"]}).to_csv(csv_path, index=False)
df_csv = SparkDataset._read_file(str(csv_path), spark)
assert df_csv.count() == 2, "Должно быть 2 строки"
assert set(df_csv.columns) == {"id", "name"}, "Колонки должны совпадать"
print("✓ Passed")

# Тест 4.2: Чтение Parquet файла
print("\n[TEST 4.2] Чтение Parquet файла")
parquet_path = Path(temp_dir) / "test.parquet"
spark.createDataFrame([(1, "X"), (2, "Y")], ["id", "name"]).write.parquet(str(parquet_path))
df_parquet = SparkDataset._read_file(str(parquet_path), spark)
assert df_parquet.count() == 2, "Должно быть 2 строки"
print("✓ Passed")

# Тест 4.3: Чтение JSON файла
print("\n[TEST 4.3] Чтение JSON файла")
json_path = Path(temp_dir) / "test.json"
pd.DataFrame({"id": [1, 2], "name": ["A", "B"]}).to_json(json_path, orient="records", lines=True)
df_json = SparkDataset._read_file(str(json_path), spark)
assert df_json.count() == 2, "Должно быть 2 строки"
print("✓ Passed")

# Тест 4.4: Чтение ORC файла (если поддерживается)
print("\n[TEST 4.4] Чтение ORC файла")
try:
    orc_path = Path(temp_dir) / "test.orc"
    spark.createDataFrame([(1, "X"), (2, "Y")], ["id", "name"]).write.orc(str(orc_path))
    df_orc = SparkDataset._read_file(str(orc_path), spark)
    assert df_orc.count() == 2, "Должно быть 2 строки"
    print("✓ Passed")
except Exception as e:
    print(f"⚠ Skipped (ORC may not be available): {e}")

# Тест 4.5: Ошибка при неподдерживаемом расширении
print("\n[TEST 4.5] Ошибка при неподдерживаемом расширении")
bad_path = Path(temp_dir) / "test.txt"
bad_path.write_text("dummy")
try:
    SparkDataset._read_file(str(bad_path), spark)
    assert False, "Должна быть ошибка ValueError"
except ValueError as e:
    assert "Unsupported file extension" in str(e)
    print("✓ Passed: ValueError raised correctly")

# Очистка
import shutil
shutil.rmtree(temp_dir)

print("\n✅ Все тесты _read_file пройдены!")

=== Тесты _read_file ===

[TEST 4.1] Чтение CSV файла
✓ Passed

[TEST 4.2] Чтение Parquet файла
✓ Passed

[TEST 4.3] Чтение JSON файла
✓ Passed

[TEST 4.4] Чтение ORC файла
✓ Passed

[TEST 4.5] Ошибка при неподдерживаемом расширении
✓ Passed: ValueError raised correctly

✅ Все тесты _read_file пройдены!


In [6]:
# ============================================================================
# ЯЧЕЙКА 5: Тесты add_row_index
# ============================================================================
print("=== Тесты add_row_index ===")

print("\n[TEST 5.1] Добавление индекса строк")
test_df = create_test_df(spark)
ds = SparkDataset(data=test_df, session=spark)
df_with_idx = ds.add_row_index(test_df, "row_idx")
assert "row_idx" in df_with_idx.columns, "Колонка row_idx должна существовать"
assert df_with_idx.count() == 5, "Количество строк не должно измениться"
# Проверяем, что индексы идут подряд
indices = [row["row_idx"] for row in df_with_idx.select("row_idx").orderBy("row_idx").collect()]
assert indices == [0, 1, 2, 3, 4], f"Индексы должны быть [0,1,2,3,4], получено {indices}"
print("✓ Passed")

print("\n[TEST 5.2] Тип индекса должен быть LongType")
schema_fields = {f.name: f.dataType for f in df_with_idx.schema.fields}
assert isinstance(schema_fields["row_idx"], T.LongType), "Тип индекса должен быть LongType"
print("✓ Passed")

print("\n✅ Все тесты add_row_index пройдены!")

=== Тесты add_row_index ===

[TEST 5.1] Добавление индекса строк
✓ Passed

[TEST 5.2] Тип индекса должен быть LongType
✓ Passed

✅ Все тесты add_row_index пройдены!


In [7]:
# ============================================================================
# ЯЧЕЙКА 6: Тесты __getitem__ и __len__
# ============================================================================
print("=== Тесты __getitem__ и __len__ ===")

print("\n[TEST 6.1] __getitem__ должен вызывать NotImplementedError")
ds = SparkDataset(data=create_test_df(spark), session=spark)
try:
    _ = ds["id"]
    assert False, "Должна быть ошибка NotImplementedError"
except NotImplementedError as e:
    print("✓ Passed: NotImplementedError raised correctly")

print("\n[TEST 6.2] __len__ возвращает количество строк")
ds = SparkDataset(data=create_test_df(spark), session=spark)
assert len(ds) == 5, f"Ожидаем 5 строк, получено {len(ds)}"
print("✓ Passed")

print("\n[TEST 6.3] __len__ для пустого DataFrame")
ds_empty = SparkDataset(data=None, session=spark)
assert len(ds_empty) == 0, "Длина пустого DataFrame должна быть 0"
print("✓ Passed")

print("\n✅ Все тесты __getitem__ и __len__ пройдены!")

=== Тесты __getitem__ и __len__ ===

[TEST 6.1] __getitem__ должен вызывать NotImplementedError
✓ Passed: NotImplementedError raised correctly

[TEST 6.2] __len__ возвращает количество строк
✓ Passed

[TEST 6.3] __len__ для пустого DataFrame
✓ Passed

✅ Все тесты __getitem__ и __len__ пройдены!


In [8]:
# ============================================================================
# ЯЧЕЙКА 7: Тесты __magic_determine_other
# ============================================================================
print("=== Тесты __magic_determine_other ===")

ds = SparkDataset(data=create_test_df(spark), session=spark)

print("\n[TEST 7.1] Преобразование SparkDataset")
other_ds = SparkDataset(data=create_test_df(spark), session=spark)
result = ds._SparkNavigation__magic_determine_other(other_ds)
assert isinstance(result, pyspark.sql.DataFrame), "Должен вернуть Spark DataFrame"
print("✓ Passed")

print("\n[TEST 7.2] Преобразование Spark DataFrame")
test_df = create_test_df(spark)
result = ds._SparkNavigation__magic_determine_other(test_df)
assert result is test_df, "Должен вернуть тот же DataFrame"
print("✓ Passed")

print("\n[TEST 7.3] Преобразование Column")
col = F.col("id")
result = ds._SparkNavigation__magic_determine_other(col)
assert isinstance(result, F.Column), "Должен вернуть Column"
print("✓ Passed")

print("\n[TEST 7.4] Преобразование примитивов (int, float, str, bool)")
for val in [42, 3.14, "test", True]:
    result = ds._SparkNavigation__magic_determine_other(val)
    assert isinstance(result, F.Column), f"Для {type(val)} должен вернуть Column"
print("✓ Passed")

print("\n[TEST 7.5] Преобразование numpy типов")
for val in [np.int32(10), np.float64(2.5), np.bool_(False)]:
    result = ds._SparkNavigation__magic_determine_other(val)
    assert isinstance(result, F.Column), f"Для {type(val)} должен вернуть Column"
print("✓ Passed")

print("\n[TEST 7.6] Ошибка для неподдерживаемого типа")
try:
    ds._SparkNavigation__magic_determine_other({"key": "value"})
    assert False, "Должна быть ошибка TypeError"
except TypeError as e:
    assert "Unsupported operand type" in str(e)
    print("✓ Passed: TypeError raised correctly")

print("\n✅ Все тесты __magic_determine_other пройдены!")

=== Тесты __magic_determine_other ===

[TEST 7.1] Преобразование SparkDataset
✓ Passed

[TEST 7.2] Преобразование Spark DataFrame
✓ Passed

[TEST 7.3] Преобразование Column
✓ Passed

[TEST 7.4] Преобразование примитивов (int, float, str, bool)
✓ Passed

[TEST 7.5] Преобразование numpy типов
✓ Passed

[TEST 7.6] Ошибка для неподдерживаемого типа
✓ Passed: TypeError raised correctly

✅ Все тесты __magic_determine_other пройдены!


In [9]:
# ============================================================================
# ЯЧЕЙКА 8: Тесты add_column
# ============================================================================
print("=== Тесты add_column ===")

print("\n[TEST 8.1] Добавление колонки из Spark DataFrame")
base_df = create_test_df(spark)
ds = SparkDataset(data=base_df, session=spark)
new_col_df = spark.createDataFrame([(10,), (20,), (30,), (40,), (50,)], ["new_col"])
result = ds.add_column(new_col_df)
assert "new_col" in result.columns, "Новая колонка должна быть добавлена"
assert len(result) == 5, "Количество строк не должно измениться"
values = [row["new_col"] for row in result.select("new_col").orderBy("id").collect()]
assert values == [10, 20, 30, 40, 50], f"Значения не совпадают: {values}"
print("✓ Passed")

print("\n[TEST 8.2] Ошибка при несовпадении количества строк в DataFrame")
bad_df = spark.createDataFrame([(10,), (20,)], ["new_col"])  # Только 2 строки
try:
    ds.add_column(bad_df)
    assert False, "Должна быть ошибка ValueError"
except ValueError as e:
    assert "Row count mismatch" in str(e)
    print("✓ Passed: ValueError raised correctly")

print("\n[TEST 8.3] Добавление колонки из списка значений")
result = ds.add_column([100, 200, 300, 400, 500], name="list_col")
assert "list_col" in result.columns, "Колонка из списка должна быть добавлена"
values = [row["list_col"] for row in result.select("list_col").orderBy("id").collect()]
assert values == [100, 200, 300, 400, 500], f"Значения не совпадают: {values}"
print("✓ Passed")

print("\n[TEST 8.4] Добавление нескольких колонок из списка списков")
result = ds.add_column(
    [[100, 200, 300, 400, 500], ["A", "B", "C", "D", "E"]], 
    name=["num_col", "str_col"]
)
assert "num_col" in result.columns and "str_col" in result.columns, "Обе колонки должны быть добавлены"
print("✓ Passed")

print("\n[TEST 8.5] Ошибка при несовпадении длины списка и количества строк")
try:
    ds.add_column([1, 2], name="short_list")
    assert False, "Должна быть ошибка ValueError"
except ValueError as e:
    assert "doesn't match DataFrame row count" in str(e)
    print("✓ Passed: ValueError raised correctly")

print("\n[TEST 8.6] Автоматическое имя колонки при name=None")
result = ds.add_column([1, 2, 3, 4, 5])
# Имя должно быть col_N где N - текущее количество колонок
expected_name = f"col_{len(base_df.columns)}"
assert expected_name in result.columns, f"Ожидаем колонку {expected_name}"
print("✓ Passed")

print("\n[TEST 8.7] Ошибка при неверном типе data")
try:
    ds.add_column("invalid_type")
    assert False, "Должна быть ошибка ValueError"
except ValueError as e:
    assert "data must be Spark DataFrame, list of values, or list of lists" in str(e)
    print("✓ Passed: ValueError raised correctly")

print("\n✅ Все тесты add_column пройдены!")

=== Тесты add_column ===

[TEST 8.1] Добавление колонки из Spark DataFrame


AttributeError: 'SparkDataset' object has no attribute 'select'

In [ ]:
len(result)

In [19]:
# ============================================================================
# ЯЧЕЙКА 9 (ИСПРАВЛЕННАЯ): Тесты операторов сравнения с numeric-only DataFrame
# ============================================================================
print("=== Тесты операторов сравнения ===")

# Создаём DataFrame ТОЛЬКО с числовыми колонками для тестов операторов
numeric_df = spark.createDataFrame([
    (1, 25, 3.14),
    (2, 30, 2.71),
    (3, 35, 1.41),
    (4, 28, 9.81),
    (5, 22, 0.57)
], ["id", "age", "score"])

ds = SparkDataset(data=numeric_df, session=spark)

for op_name, op_symbol, test_val, expected_id in [
    ("__eq__", "==", 1, [True, False, False, False, False]),
    ("__ne__", "!=", 1, [False, True, True, True, True]),
    ("__lt__", "<", 3, [True, True, False, False, False]),
    ("__le__", "<=", 3, [True, True, True, False, False]),
    ("__ge__", ">=", 3, [False, False, True, True, True]),
    ("__gt__", ">", 3, [False, False, False, True, True]),
]:
    print(f"\n[TEST 9.{op_name}] Оператор {op_symbol} с test_val={test_val}")
    result_df = getattr(ds, op_name)(test_val)
    
    # Берём значения из колонки "id" (первая колонка)
    results = [row["id"] for row in result_df["id"].collect()]
    
    assert results == expected_id, f"Ожидаем {expected_id}, получено {results}"
    print(f"✓ Passed: {results}")

print("\n✅ Все тесты операторов сравнения пройдены!")

=== Тесты операторов сравнения ===

[TEST 9.__eq__] Оператор == с test_val=1


NotImplementedError: Spark-base Dataset does not support indexing

In [11]:
# ============================================================================
# ЯЧЕЙКА 10 (ИСПРАВЛЕННАЯ): Тесты унарных операторов с numeric-only DataFrame
# ============================================================================
print("=== Тесты унарных операторов ===")

# Создаём DataFrame ТОЛЬКО с числовыми данными для тестов унарных операторов
numeric_df = spark.createDataFrame([
    (1, -2.5, 10),
    (2, 3.7, -20),
    (3, -0.1, 30)
], ["id", "value", "score"])

ds = SparkDataset(data=numeric_df, session=spark)

print("\n[TEST 10.1] __pos__ (унарный плюс)")
result = +ds
assert result.collect() == numeric_df.collect(), "Унарный плюс должен вернуть те же данные"
print("✓ Passed")

print("\n[TEST 10.2] __neg__ (унарный минус)")
result = -ds
expected_values = [-1, -2, -3]  # Для колонки id
actual_values = [row["id"] for row in result.select("id").collect()]
assert actual_values == expected_values, f"Ожидаем {expected_values}, получено {actual_values}"
print("✓ Passed")

print("\n[TEST 10.3] __abs__ (модуль)")
result = abs(ds)
expected_values = [2.5, 3.7, 0.1]  # Для колонки value
actual_values = [row["value"] for row in result.select("value").collect()]
assert actual_values == expected_values, f"Ожидаем {expected_values}, получено {actual_values}"
print("✓ Passed")

print("\n[TEST 10.4] __round__ (округление)")
ds_float = SparkDataset(
    data=spark.createDataFrame([(1.234,), (2.567,), (3.891,)], ["val"]), 
    session=spark
)
result = round(ds_float, 1)
expected = [1.2, 2.6, 3.9]
actual = [row["val"] for row in result.select("val").collect()]
assert actual == expected, f"Ожидаем {expected}, получено {actual}"
print("✓ Passed")

print("\n[TEST 10.5] __invert__ (логическое НЕ) - только для boolean колонок")
bool_df = spark.createDataFrame([
    (True,),
    (False,),
    (True,)
], ["flag"])
ds_bool = SparkDataset(data=bool_df, session=spark)
result = ~ds_bool
expected_flags = [False, True, False]
actual_flags = [row["flag"] for row in result.select("flag").collect()]
assert actual_flags == expected_flags, f"Ожидаем {expected_flags}, получено {actual_flags}"
print("✓ Passed")

print("\n✅ Все тесты унарных операторов пройдены!")

=== Тесты унарных операторов ===

[TEST 10.1] __pos__ (унарный плюс)


AttributeError: 'SparkDataset' object has no attribute 'collect'

In [12]:
# ============================================================================
# ЯЧЕЙКА 11: Тесты бинарных арифметических операторов
# ============================================================================
print("=== Тесты бинарных арифметических операторов ===")

ds = SparkDataset(data=spark.createDataFrame([(10,), (20,), (30,)], ["val"]), session=spark)

test_cases = [
    ("__add__", "+", 5, [15, 25, 35]),
    ("__sub__", "-", 5, [5, 15, 25]),
    ("__mul__", "*", 2, [20, 40, 60]),
    ("__truediv__", "/", 2, [5.0, 10.0, 15.0]),
    ("__floordiv__", "//", 3, [3, 6, 10]),
    ("__mod__", "%", 7, [3, 6, 2]),
    ("__pow__", "**", 2, [100, 400, 900]),
]

for op_name, symbol, operand, expected in test_cases:
    print(f"\n[TEST 11.{op_name}] Оператор {symbol} с operand={operand}")
    result = getattr(ds, op_name)(operand)
    actual = [row[0] for row in result.select("val").collect()]
    assert actual == expected, f"Ожидаем {expected}, получено {actual}"
    print(f"✓ Passed")

print("\n[TEST 11.__div__] Оператор __div__ (alias для __truediv__)")
result = ds.__div__(2)
expected = [5.0, 10.0, 15.0]
actual = [row[0] for row in result.select("val").collect()]
assert actual == expected, f"Ожидаем {expected}, получено {actual}"
print("✓ Passed")

print("\n✅ Все тесты бинарных арифметических операторов пройдены!")

=== Тесты бинарных арифметических операторов ===

[TEST 11.__add__] Оператор + с operand=5


AttributeError: 'SparkDataset' object has no attribute 'select'

In [13]:
# ============================================================================
# ЯЧЕЙКА 12 (ИСПРАВЛЕННАЯ): Тесты бинарных логических операторов
# ============================================================================
print("=== Тесты бинарных логических операторов ===")

# DataFrame ТОЛЬКО с boolean колонками
bool_df = spark.createDataFrame([
    (True, True),
    (True, False),
    (False, True),
    (False, False)
], ["a", "b"])
ds = SparkDataset(data=bool_df, session=spark)

print("\n[TEST 12.1] __and__ (логическое И)")
result = ds.__and__(F.col("b"))
expected = [True, False, False, False]
actual = [row["a"] for row in result.select("a").collect()]
assert actual == expected, f"Ожидаем {expected}, получено {actual}"
print("✓ Passed")

print("\n[TEST 12.2] __or__ (логическое ИЛИ)")
result = ds.__or__(F.col("b"))
expected = [True, True, True, False]
actual = [row["a"] for row in result.select("a").collect()]
assert actual == expected, f"Ожидаем {expected}, получено {actual}"
print("✓ Passed")

print("\n[TEST 12.3] __xor__ (исключающее ИЛИ)")
try:
    result = ds.__xor__(F.col("b"))
    expected = [False, True, True, False]
    actual = [row["a"] for row in result.select("a").collect()]
    assert actual == expected, f"Ожидаем {expected}, получено {actual}"
    print("✓ Passed")
except Exception as e:
    print(f"⚠ XOR может не поддерживаться в вашей версии Spark: {type(e).__name__}")
    # Альтернативная проверка через ручную логику
    result_alt = ds.data.select([
        ((F.col("a") & ~F.col("b")) | (~F.col("a") & F.col("b"))).alias("a")
    ])
    actual = [row["a"] for row in result_alt.collect()]
    expected = [False, True, True, False]
    assert actual == expected, f"Ожидаем {expected}, получено {actual}"
    print("✓ Passed (alternative implementation)")

print("\n✅ Все тесты бинарных логических операторов пройдены!")

=== Тесты бинарных логических операторов ===

[TEST 12.1] __and__ (логическое И)


AttributeError: 'SparkDataset' object has no attribute 'select'

In [14]:
# ============================================================================
# ЯЧЕЙКА 13: Тесты обратных арифметических операторов
# ============================================================================
print("=== Тесты обратных арифметических операторов ===")

ds = SparkDataset(data=spark.createDataFrame([(10,), (20,), (30,)], ["val"]), session=spark)

test_cases = [
    ("__radd__", 5, [15, 25, 35]),  # 5 + val
    ("__rmul__", 2, [20, 40, 60]),  # 2 * val
    ("__rsub__", 100, [90, 80, 70]),  # 100 - val
    ("__rtruediv__", 100, [10.0, 5.0, 100/30]),  # 100 / val
    ("__rfloordiv__", 100, [10, 5, 3]),  # 100 // val
    ("__rmod__", 100, [0, 0, 10]),  # 100 % val
    ("__rpow__", 2, [1024, 2**20, 2**30]),  # 2 ** val (большие числа!)
]

for op_name, operand, expected in test_cases:
    print(f"\n[TEST 13.{op_name}] Обратный оператор с operand={operand}")
    # Для __rpow__ ожидаем очень большие числа, проверяем только первые два
    if op_name == "__rpow__":
        result = getattr(ds, op_name)(2)
        actual = [row[0] for row in result.select("val").collect()]
        assert actual[0] == 1024 and actual[1] == 2**20, f"Проверка __rpow__ не пройдена"
    else:
        result = getattr(ds, op_name)(operand)
        actual = [row[0] for row in result.select("val").collect()]
        # Округляем для сравнения float
        if any(isinstance(x, float) for x in expected):
            actual = [round(x, 5) for x in actual]
            expected = [round(x, 5) for x in expected]
        assert actual == expected, f"Ожидаем {expected}, получено {actual}"
    print(f"✓ Passed")

print("\n[TEST 13.__rdiv__] Обратный __div__ (alias для __rtruediv__)")
result = ds.__rdiv__(100)
expected = [10.0, 5.0, 100/30]
actual = [round(x, 5) for x in [row[0] for row in result.select("val").collect()]]
expected = [round(x, 5) for x in expected]
assert actual == expected, f"Ожидаем {expected}, получено {actual}"
print("✓ Passed")

print("\n✅ Все тесты обратных арифметических операторов пройдены!")

=== Тесты обратных арифметических операторов ===

[TEST 13.__radd__] Обратный оператор с operand=5


AttributeError: 'SparkDataset' object has no attribute 'select'

In [15]:
# ============================================================================
# ЯЧЕЙКА 14 (ИСПРАВЛЕННАЯ): Тесты методов представления
# ============================================================================
print("=== Тесты методов представления ===")

print("\n[TEST 14.1] __repr__ для непустого DataFrame")
ds = SparkDataset(data=create_test_df(spark), session=spark)
repr_str = repr(ds)
assert "SparkNavigation" in repr_str, "repr должен содержать имя класса"
assert "rows=5" in repr_str, "repr должен содержать количество строк"
assert "columns=5" in repr_str, "repr должен содержать количество колонок"
print(f"repr output: {repr_str[:100]}...")
print("✓ Passed")

print("\n[TEST 14.2] __repr__ для пустого DataFrame")
ds_empty = SparkDataset(data=None, session=spark)
repr_str = repr(ds_empty)
# Пустой DataFrame имеет rows=0, columns=0 (а не data=None)
assert "rows=0" in repr_str, f"repr для пустого должен содержать rows=0, получено: {repr_str}"
assert "columns=0" in repr_str, f"repr для пустого должен содержать columns=0, получено: {repr_str}"
print(f"repr output: {repr_str}")
print("✓ Passed")

print("\n[TEST 14.3] _repr_html_ для непустого DataFrame")
ds = SparkDataset(data=create_test_df(spark), session=spark)
html_str = ds._repr_html_()
assert "<p" in html_str or "<div" in html_str, "HTML представление должно содержать теги"
assert "SparkNavigation" in html_str, "HTML должен содержать имя класса"
print("✓ Passed")

print("\n[TEST 14.4] _repr_html_ для пустого DataFrame")
ds_empty = SparkDataset(data=None, session=spark)
html_str = ds_empty._repr_html_()
# Для пустого DataFrame HTML должен отображать rows=0
assert "0 rows" in html_str or "rows=0" in html_str or "0 × 0" in html_str, \
    f"HTML для пустого должен содержать информацию о 0 строках, получено: {html_str}"
print(f"HTML output: {html_str[:200]}...")
print("✓ Passed")

print("\n✅ Все тесты методов представления пройдены!")

=== Тесты методов представления ===

[TEST 14.1] __repr__ для непустого DataFrame
repr output: SparkNavigation(rows=5, columns=5, schema=[id:int, name:string, age:int, score:double, is_active:boo...
✓ Passed

[TEST 14.2] __repr__ для пустого DataFrame
repr output: SparkNavigation(rows=0, columns=0, schema=[])
✓ Passed

[TEST 14.3] _repr_html_ для непустого DataFrame
✓ Passed

[TEST 14.4] _repr_html_ для пустого DataFrame
HTML output: 
            <div style="margin: 10px 0;">
                <p style="font-family: monospace; font-size: 12px; color: #666;">
                    SparkNavigation: 0 rows × 0 columns
                </p...
✓ Passed

✅ Все тесты методов представления пройдены!


In [ ]:
# ============================================================================
# ЯЧЕЙКА 15: Тесты метода get
# ============================================================================
print("=== Тесты метода get ===")

ds = SparkDataset(data=create_test_df(spark), session=spark)

print("\n[TEST 15.1] get по имени колонки (str)")
result = ds.get("name")
assert result is not None, "Результат не должен быть None"
assert "name" in result.columns, "Результат должен содержать колонку name"
assert result.count() == 5, "Должно быть 5 строк"
print("✓ Passed")

print("\n[TEST 15.2] get по списку колонок")
result = ds.get(["id", "age"])
assert set(result.columns) == {"id", "age"}, "Должны быть только указанные колонки"
print("✓ Passed")

print("\n[TEST 15.3] get по несуществующей колонке с default")
result = ds.get("nonexistent", default="DEFAULT")
assert result == "DEFAULT", f"Ожидаем DEFAULT, получено {result}"
print("✓ Passed")

print("\n[TEST 15.4] get по индексу строки (int)")
result = ds.get(2)  # Третья строка (0-based)
assert result.count() == 1, "Должна быть одна строка"
row = result.first()
assert row["id"] == 3, f"Ожидаем id=3, получено {row['id']}"  # Charlie
print("✓ Passed")

print("\n[TEST 15.5] get по списку индексов строк")
result = ds.get([0, 2, 4])
assert result.count() == 3, "Должно быть 3 строки"
ids = [row["id"] for row in result.select("id").collect()]
assert ids == [1, 3, 5], f"Ожидаем [1,3,5], получено {ids}"
print("✓ Passed")

print("\n[TEST 15.6] get по tuple (row_idx, column_name)")
result = ds.get((2, "name"))  # Третья строка, колонка name
assert result == "Charlie", f"Ожидаем 'Charlie', получено {result}"
print("✓ Passed")

print("\n[TEST 15.7] get по tuple с несуществующей колонкой")
result = ds.get((2, "nonexistent"), default="NOT_FOUND")
assert result == "NOT_FOUND", f"Ожидаем NOT_FOUND, получено {result}"
print("✓ Passed")

print("\n[TEST 15.8] get с пустым ключом")
result = ds.get([], default="EMPTY")
assert result == "EMPTY", f"Ожидаем EMPTY, получено {result}"
print("✓ Passed")

print("\n✅ Все тесты метода get пройдены!")

In [ ]:
# ============================================================================
# ЯЧЕЙКА 16: Тесты метода take
# ============================================================================
print("=== Тесты метода take ===")

ds = SparkDataset(data=create_test_df(spark), session=spark)

print("\n[TEST 16.1] take по индексу колонки (axis=1)")
result = ds.take(1, axis=1)  # Вторая колонка: name
assert result.count() == 5, "Должно быть 5 строк"
assert result.columns == ["name"], f"Ожидаем ['name'], получено {result.columns}"
print("✓ Passed")

print("\n[TEST 16.2] take по списку индексов колонок")
result = ds.take([0, 2], axis="columns")  # id и age
assert set(result.columns) == {"id", "age"}, "Должны быть только указанные колонки"
print("✓ Passed")

print("\n[TEST 16.3] take с выходом за границы колонок")
try:
    ds.take(10, axis=1)
    assert False, "Должна быть ошибка IndexError"
except IndexError as e:
    assert "out of range" in str(e)
    print("✓ Passed: IndexError raised correctly")

print("\n[TEST 16.4] take по индексу строки (axis=0)")
result = ds.take(2, axis=0)  # Третья строка
assert result.count() == 1, "Должна быть одна строка"
row = result.first()
assert row["id"] == 3, f"Ожидаем id=3, получено {row['id']}"
print("✓ Passed")

print("\n[TEST 16.5] take по списку индексов строк")
result = ds.take([0, 4], axis="rows")
assert result.count() == 2, "Должно быть 2 строки"
ids = [row["id"] for row in result.select("id").collect()]
assert ids == [1, 5], f"Ожидаем [1,5], получено {ids}"
print("✓ Passed")

print("\n[TEST 16.6] take с невалидным axis")
try:
    ds.take(0, axis="invalid")
    assert False, "Должна быть ошибка ValueError"
except ValueError as e:
    assert "Invalid axis" in str(e)
    print("✓ Passed: ValueError raised correctly")

print("\n✅ Все тесты метода take пройдены!")

In [ ]:
# ============================================================================
# ЯЧЕЙКА 17: Тесты методов get_values и iget_values
# ============================================================================
print("=== Тесты get_values и iget_values ===")

ds = SparkDataset(data=create_test_df(spark), session=spark)

print("\n[TEST 17.1] get_values с row и column")
result = ds.get_values(row=2, column="name")  # Третья строка, колонка name
assert result == "Charlie", f"Ожидаем 'Charlie', получено {result}"
print("✓ Passed")

print("\n[TEST 17.2] get_values с несуществующей колонкой")
try:
    ds.get_values(column="nonexistent")
    assert False, "Должна быть ошибка ValueError"
except ValueError as e:
    assert "not found" in str(e)
    print("✓ Passed: ValueError raised correctly")

print("\n[TEST 17.3] get_values только по column (возвращает список)")
result = ds.get_values(column="id")
assert isinstance(result, list), "Должен вернуть список"
assert result == [1, 2, 3, 4, 5], f"Ожидаем [1,2,3,4,5], получено {result}"
print("✓ Passed")

print("\n[TEST 17.4] get_values только по row (возвращает список значений строки)")
result = ds.get_values(row=0)
assert isinstance(result, list), "Должен вернуть список"
assert result[0] == 1, f"Первое значение должно быть 1, получено {result[0]}"  # id
assert result[1] == "Alice", f"Второе значение должно быть 'Alice', получено {result[1]}"  # name
print("✓ Passed")

print("\n[TEST 17.5] get_values без параметров (возвращает self)")
result = ds.get_values()
assert result is ds, "Должен вернуть self"
print("✓ Passed")

print("\n[TEST 17.6] iget_values по индексам")
result = ds.iget_values(row=2, column=1)  # Третья строка, вторая колонка (name)
assert result == "Charlie", f"Ожидаем 'Charlie', получено {result}"
print("✓ Passed")

print("\n[TEST 17.7] iget_values с выходом за границы колонки")
try:
    ds.iget_values(column=10)
    assert False, "Должна быть ошибка IndexError"
except IndexError as e:
    assert "out of range" in str(e)
    print("✓ Passed: IndexError raised correctly")

print("\n✅ Все тесты get_values и iget_values пройдены!")

In [16]:
# ============================================================================
# ЯЧЕЙКА 18: Тесты метода create_empty
# ============================================================================
print("=== Тесты create_empty ===")

print("\n[TEST 18.1] create_empty с columns")
ds = SparkDataset(data=None, session=spark)
result = ds.create_empty(columns=["col1", "col2"])
assert result.data.count() == 0, "Должен быть пустой DataFrame"
assert set(result.data.columns) == {"col1", "col2"}, "Должны быть указанные колонки"
print("✓ Passed")

print("\n[TEST 18.2] create_empty с index и columns")
result = ds.create_empty(index=["a", "b", "c"], columns=["col1"])
assert result.data.count() == 3, "Должно быть 3 строки"
assert "index" in result.data.columns, "Должна быть колонка index"
assert "col1" in result.data.columns, "Должна быть колонка col1"
# Проверяем, что значения колонок - None
null_count = result.data.filter(F.col("col1").isNotNull()).count()
assert null_count == 0, "Все значения должны быть None"
print("✓ Passed")

print("\n[TEST 18.3] create_empty без параметров")
result = ds.create_empty()
assert result.data.count() == 0, "Должен быть пустой DataFrame"
assert len(result.data.columns) == 0, "Не должно быть колонок"
print("✓ Passed")

print("\n[TEST 18.4] create_empty с разными типами index")
for idx_val, expected_type in [
    (True, T.BooleanType),
    (42, T.LongType),
    (3.14, T.DoubleType),
    ("str", T.StringType)
]:
    result = ds.create_empty(index=[idx_val], columns=["test"])
    schema_types = {f.name: type(f.dataType) for f in result.data.schema.fields}
    assert schema_types["index"] == expected_type, f"Тип index должен быть {expected_type}"
print("✓ Passed")

print("\n✅ Все тесты create_empty пройдены!")

=== Тесты create_empty ===

[TEST 18.1] create_empty с columns
✓ Passed

[TEST 18.2] create_empty с index и columns
✓ Passed

[TEST 18.3] create_empty без параметров
✓ Passed

[TEST 18.4] create_empty с разными типами index
✓ Passed

✅ Все тесты create_empty пройдены!


In [17]:
# ============================================================================
# ЯЧЕЙКА 19: Итоговый сводный тест
# ============================================================================
print("=== 🎉 ИТОГОВЫЙ СВОДНЫЙ ТЕСТ ===\n")

# Создаем тестовый датасет
test_df = create_test_df(spark)
ds = SparkDataset(data=test_df, session=spark)

# Проверяем основные операции в цепочке
print("[FINAL TEST] Цепочка операций:")

# 1. Добавляем колонку
ds_with_new = ds.add_column([10, 20, 30, 40, 50], name="bonus")
print(f"  ✓ Добавлена колонка 'bonus', колонок стало: {len(ds_with_new.columns)}")

# 2. Применяем арифметическую операцию
ds_modified = ds_with_new + 5
print(f"  ✓ Применено +5 ко всем числовым колонкам")

# 3. Фильтруем через get
subset = ds.get([0, 2, 4])
print(f"  ✓ Выбраны строки [0, 2, 4], строк: {subset.count()}")

# 4. Проверяем представление
repr_ok = "SparkNavigation" in repr(ds)
print(f"  ✓ __repr__ работает: {repr_ok}")

# 5. Проверяем длину
len_ok = len(ds) == 5
print(f"  ✓ __len__ работает: {len_ok}")

print("\n" + "="*60)
print("✅ ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО! 🚀")
print("="*60)

=== 🎉 ИТОГОВЫЙ СВОДНЫЙ ТЕСТ ===

[FINAL TEST] Цепочка операций:
  ✓ Добавлена колонка 'bonus', колонок стало: 6


AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_DIFF_TYPES] Cannot resolve "(is_active + 5)" due to data type mismatch: the left and right operands of the binary operator have incompatible types ("BOOLEAN" and "INT").;
'Project [(id#467L + cast(5 as bigint)) AS id#479L, (cast(name#468 as double) + cast(5 as double)) AS name#480, (age#469L + cast(5 as bigint)) AS age#481L, (score#470 + cast(5 as double)) AS score#482, (is_active#471 + 5) AS is_active#483, (bonus#472L + cast(5 as bigint)) AS bonus#484L]
+- LogicalRDD [id#467L, name#468, age#469L, score#470, is_active#471, bonus#472L], false


In [18]:
ds_with_new

id,name,age,score,is_active,bonus
1,Alice,25,3.14,True,10
2,Bob,30,2.71,False,20
3,Charlie,35,1.41,True,30
4,Diana,28,9.81,False,40
5,Eve,22,0.57,True,50


In [ ]:
from __future__ import annotations

from typing import Any, Dict, Type, Union
import numpy as np
from pyspark.sql.types import (
    DataType,
    StringType,
    BinaryType,
    BooleanType,
    LongType,
    IntegerType,
    ShortType,
    ByteType,
    DoubleType,
    FloatType,
    DateType,
    TimestampType,
    NullType,
    ArrayType,
    StructType,
    MapType,
)


class SparkTypeMapper:
    _PY_TO_SPARK: Dict[Type, DataType] = {
        # Strings
        str: StringType(),
        bytes: BinaryType(),
        
        # Boolean
        bool: BooleanType(),
        np.bool_: BooleanType(),
        
        # Integers
        int: LongType(),
        np.integer: LongType(),
        np.int64: LongType(),
        np.int32: IntegerType(),
        np.int16: ShortType(),
        np.int8: ByteType(),
        
        float: DoubleType(),
        np.floating: DoubleType(),
        np.float64: DoubleType(),
        np.float32: FloatType(),
        
        # datetime.date: DateType(),
        # datetime.datetime: TimestampType(),
        
        # Special cases
        type(None): NullType(),  # None → NullType (более семантично, чем StringType)
        object: StringType(),    # Fallback для неизвестных объектов
    }

    @classmethod
    def to_spark(cls, value: Any | Type) -> DataType:
        if isinstance(value, type):
            return cls._resolve_python_type(value)
        return cls._resolve_python_value(value)

    @classmethod
    def _resolve_python_type(cls, py_type: Type) -> DataType:
        if py_type in cls._PY_TO_SPARK:
            return cls._PY_TO_SPARK[py_type]
        
        for base_type, spark_type in cls._PY_TO_SPARK.items():
            if isinstance(base_type, type) and base_type is not type(None):
                try:
                    if issubclass(py_type, base_type):
                        return spark_type
                except TypeError:
                    continue
        return StringType()

    @classmethod
    def _resolve_python_value(cls, value: Any) -> DataType:
        if value is None:
            return cls._PY_TO_SPARK[type(None)]
        
        if isinstance(value, bool):
            return cls._PY_TO_SPARK[bool]
        if isinstance(value, np.bool_):
            return cls._PY_TO_SPARK[np.bool_]
        
        if isinstance(value, np.integer):
            if isinstance(value, np.int8):
                return cls._PY_TO_SPARK[np.int8]
            elif isinstance(value, np.int16):
                return cls._PY_TO_SPARK[np.int16]
            elif isinstance(value, np.int32):
                return cls._PY_TO_SPARK[np.int32]
            elif isinstance(value, np.int64):
                return cls._PY_TO_SPARK[np.int64]
            else:
                return cls._PY_TO_SPARK[np.integer]
        
        if isinstance(value, np.floating):
            if isinstance(value, np.float32):
                return cls._PY_TO_SPARK[np.float32]
            elif isinstance(value, np.float64):
                return cls._PY_TO_SPARK[np.float64]
            else:
                return cls._PY_TO_SPARK[np.floating]
        
        py_type = type(value)
        return cls._resolve_python_type(py_type)

    _SPARK_TO_PY: Dict[str, Type] = {
        # Integers
        "tinyint": int,
        "byte": int,
        "smallint": int,
        "short": int,
        "int": int,
        "integer": int,
        "bigint": int,
        "long": int,
        
        # Floats
        "float": float,
        "real": float,
        "double": float,
        "decimal": float,  # Decimal → float (упрощённо, можно заменить на decimal.Decimal)
        "numeric": float,
        
        # Strings
        "string": str,
        "varchar": str,
        "char": str,
        "text": str,
        
        # Boolean
        "boolean": bool,
        "bool": bool,
        
        # Date/Time → str (безопасный дефолт; можно заменить на datetime.date/datetime)
        "date": str,
        "timestamp": str,
        "timestamp_ntz": str,
        
        # Complex types
        "array": list,
        "struct": dict,
        "map": dict,
        "object": object,
        
        # Binary
        "binary": bytes,
        "blob": bytes,
        
        # Null
        "null": type(None),
    }

    @classmethod
    def to_python(cls, spark_type: DataType | str) -> Type:
        if isinstance(spark_type, DataType):
            type_str = spark_type.simpleString()
        elif isinstance(spark_type, str):
            type_str = spark_type.strip().lower()
        else:
            raise TypeError(
                f"spark_type must be DataType or str, got {type(spark_type).__name__}"
            )
        base_type = type_str.split("(")[0].split("<")[0].strip().lower()
        
        return cls._SPARK_TO_PY.get(base_type, object)

    @classmethod
    def get_spark_type_name(cls, spark_type: DataType) -> str:
        return spark_type.simpleString().split("(")[0].split("<")[0].strip().lower()

    @classmethod
    def is_numeric(cls, spark_type: DataType | str) -> bool:
        py_type = cls.to_python(spark_type)
        return py_type in (int, float)

    @classmethod
    def is_string(cls, spark_type: DataType | str) -> bool:
        py_type = cls.to_python(spark_type)
        return py_type is str

    @classmethod
    def is_complex(cls, spark_type: DataType | str) -> bool:
        if isinstance(spark_type, DataType):
            type_str = spark_type.simpleString().lower()
        else:
            type_str = str(spark_type).strip().lower()
        
        base = type_str.split("(")[0].split("<")[0].strip()
        return base in ("array", "struct", "map")

    @classmethod
    def is_nullable(cls, spark_type: DataType | str) -> bool:
        if isinstance(spark_type, DataType):
            return getattr(spark_type, 'containsNull', True) or isinstance(spark_type, NullType)
        py_type = cls.to_python(spark_type)
        return py_type is type(None)

    @classmethod
    def validate_cast(cls, from_type: DataType | str, to_type: DataType | str) -> bool:
        from_py = cls.to_python(from_type)
        to_py = cls.to_python(to_type)
        
        if to_py is str:
            return True
        
        if from_py is str and to_py in (int, float):
            return True
        
        if from_py in (int, float) and to_py in (int, float):
            return True
        
        if from_py is bool and to_py in (int, float):
            return True
        if from_py in (int, float) and to_py is bool:
            return True

        return from_py == to_py

    @classmethod
    def get_compatible_type(cls, types: list[DataType | str | Type]) -> DataType:
        if not types:
            return NullType()
        
        if len(types) == 1:
            t = types[0]
            if isinstance(t, type):
                return cls._resolve_python_type(t)
            elif isinstance(t, DataType):
                return t
            else:
                return cls.to_spark(t)
        
        py_types = set()
        for t in types:
            if isinstance(t, type):
                py_types.add(t)
            elif isinstance(t, DataType):
                type_name = cls.get_spark_type_name(t)
                py_types.add(cls._SPARK_TO_PY.get(type_name, object))
            elif isinstance(t, str):
                py_types.add(cls.to_python(t))
            else:
                if t is None:
                    py_types.add(type(None))
                else:
                    py_types.add(type(t))
        
        if str in py_types:
            return StringType()
        
        if float in py_types:
            return DoubleType()
        
        if bool in py_types:
            if py_types <= {bool, type(None)}:
                return BooleanType()
            if py_types & {int, float}:
                return LongType() if float not in py_types else DoubleType()
        
        if int in py_types:
            return LongType()
        
        if py_types <= {type(None)}:
            return NullType()
        return StringType()

In [6]:
# Определение типа по значению
SparkTypeMapper.to_spark(np.int32(42))      # IntegerType()

IntegerType()

In [8]:
SparkTypeMapper.to_spark(None)              # NullType()

NullType()

In [9]:
SparkTypeMapper.to_spark("hello")           # StringType()

StringType()

In [10]:
# Обратное преобразование
SparkTypeMapper.to_python("bigint")         # int

int

In [11]:
SparkTypeMapper.to_python(LongType())       # int

int

In [12]:
# Валидация кастинга
SparkTypeMapper.validate_cast("string", "int")    # True (но может упасть в runtime)

True

In [13]:
SparkTypeMapper.validate_cast("int", "string")    # True

True

In [14]:
# Подбор совместимого типа для смешанных данных
SparkTypeMapper.get_compatible_type([int, float, None])  # DoubleType()

DoubleType()